In [1]:
import os, re, subprocess, sys

src = None
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = [d for d in dirs if d != "competitions"]
    if "03_preprocess_mri.py" in files:
        src = os.path.join(root, "03_preprocess_mri.py"); break
assert src, "03_preprocess_mri.py tidak ditemukan; tambahkan input rsna-knee-02-preprocess-mri"
s = open(src).read()
print("Script asli:", src)

def rep(a, b):
    global s
    assert s.count(a) == 1, f"pola tidak ditemukan tepat satu kali: {a[:60]!r}"
    s = s.replace(a, b)

tag = "crop280"
rep('OUT_DIR = f"/kaggle/working/vol_{SPLIT}"', f'OUT_DIR = f"/kaggle/working/vol_{{SPLIT}}_{tag}"')
s, n = re.subn(r"LIMIT\s*=\s*\w+", "LIMIT   = None", s, count=1)
assert n == 1, "baris LIMIT tidak ditemukan"
rep('meta_path = os.path.join(OUT_DIR, "_meta.csv")', f'meta_path = os.path.join(OUT_DIR, "_meta_{tag}.csv")')
rep('IMG     = 224', 'IMG     = 280')
# perbesar ke 360 lalu ambil bagian tengah 280 x 280 (± 78% bidang pandang, sama dengan crop)
rep('    return cv2.resize(canvas, (size, size), interpolation=cv2.INTER_AREA)',
    '    big = cv2.resize(canvas, (360, 360), interpolation=cv2.INTER_AREA)\n'
    '    o = (360 - size) // 2\n'
    '    return big[o:o + size, o:o + size]')

for line in s.splitlines():
    if line.startswith(("OUT_DIR", "IMG ", "LIMIT", "DEPTH")) or "big =" in line or "meta_path =" in line:
        print("  ", line.strip())

path = f"/kaggle/working/03_preprocess_{tag}.py"
open(path, "w").write(s)
print("Script dibuat:", path)
r = subprocess.run([sys.executable, "-u", path], capture_output=True, text=True)
print(r.stdout[-2500:]); print(r.stderr[-2000:])

Script asli: /kaggle/input/notebooks/kellenvalerie/rsna-knee-02-preprocess-mri/03_preprocess_mri.py
   OUT_DIR = f"/kaggle/working/vol_{SPLIT}_crop280"
   IMG     = 280                         # ukuran in-plane
   DEPTH   = 16                          # jumlah slice per view
   LIMIT   = None                        # None = semua studi
   big = cv2.resize(canvas, (360, 360), interpolation=cv2.INTER_AREA)
   meta_path = os.path.join(OUT_DIR, "_meta_crop280.csv")
Script dibuat: /kaggle/working/03_preprocess_crop280.py
Diproses: 4407 studi, CPU: 4

Selesai 4407 studi dalam 37.5 menit (0.51 detik/studi)
View tersedia (%):
view
Axial       100.0
Coronal     100.0
Sagittal    100.0
Jumlah error: 0
Ukuran folder output: 11.85 GB

37:00<00:31,  1.40it/s]
100%|██████████| 4407/4407 [37:29<00:00,  1.96it/s]

